# 01 — Extracción, Infraestructura y Auditoría

Este notebook corresponde a la **Persona 1**. Descarga los Parquet, crea la estructura `raw`, lee cada archivo individualmente con Spark y genera `audit_file_inventory`.

In [1]:
%pip install -q pyspark pyyaml requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sys
from pathlib import Path

# Ubicación del proyecto. En Colab puedes ajustar esta ruta si subes la carpeta a Drive.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

print("Proyecto:", PROJECT_ROOT)

Proyecto: c:\Users\saray\OneDrive\Documents\6to SEMESTRE\MODELADO\PARCIAL 2\PROYECTO\etl_spark_parquet_advanced


In [3]:
from utils import load_config, ensure_directories, generate_process_id, get_spark_session
from extract import download_nyc_files, download_apache_bad_data, build_audit_file_inventory, write_audit_inventory

config = load_config("config/etl_config.yaml")
ensure_directories(config)
process_id = generate_process_id()

print("process_id:", process_id)

process_id: 5c2747e7-dee3-467d-8766-55dafca011c7


## 1. Descarga de archivos NYC TLC

Los archivos se almacenan sin modificación en `data/raw/` respetando particiones `year=` y `month=`.

In [4]:
# Esta celda puede tardar por el tamaño de los archivos.
download_results = download_nyc_files(config, overwrite=False)
download_results[:3], len(download_results)

([{'file_name': 'yellow_tripdata_2023-01.parquet',
   'service_type': 'yellow',
   'year': 2023,
   'month': 1,
   'url': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet',
   'target_path': 'C:\\Users\\saray\\OneDrive\\Documents\\6to SEMESTRE\\MODELADO\\PARCIAL 2\\PROYECTO\\etl_spark_parquet_advanced\\data\\raw\\yellow\\year=2023\\month=01\\yellow_tripdata_2023-01.parquet',
   'download_status': 'SKIPPED_EXISTS',
   'sha256_file': '32df6f67578fa86c484a6b5ef23a5281992ff085521082340b0f9e5889e9a572'},
  {'file_name': 'yellow_tripdata_2023-02.parquet',
   'service_type': 'yellow',
   'year': 2023,
   'month': 2,
   'url': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet',
   'target_path': 'C:\\Users\\saray\\OneDrive\\Documents\\6to SEMESTRE\\MODELADO\\PARCIAL 2\\PROYECTO\\etl_spark_parquet_advanced\\data\\raw\\yellow\\year=2023\\month=02\\yellow_tripdata_2023-02.parquet',
   'download_status': 'SKIPPED_EXISTS',
   'sha256_fi

## 2. Descarga de archivos problemáticos Apache Parquet Testing

Se descargan archivos desde la carpeta `bad_data` para probar errores de lectura y aislamiento de corruptos.

In [5]:
bad_results = download_apache_bad_data(config, overwrite=False)
bad_results[:5], len(bad_results)

([{'file_name': 'ARROW-GH-41317.parquet',
   'url': 'https://raw.githubusercontent.com/apache/parquet-testing/master/bad_data/ARROW-GH-41317.parquet',
   'target_path': 'C:\\Users\\saray\\OneDrive\\Documents\\6to SEMESTRE\\MODELADO\\PARCIAL 2\\PROYECTO\\etl_spark_parquet_advanced\\data\\raw\\bad_parquet\\ARROW-GH-41317.parquet',
   'download_status': 'SKIPPED_EXISTS',
   'sha256_file': 'a9c2b53992b6c017d3cbe44b9d3a0b2ebbb0f68c8eda424f11942d3313bd5696'},
  {'file_name': 'ARROW-GH-41321.parquet',
   'url': 'https://raw.githubusercontent.com/apache/parquet-testing/master/bad_data/ARROW-GH-41321.parquet',
   'target_path': 'C:\\Users\\saray\\OneDrive\\Documents\\6to SEMESTRE\\MODELADO\\PARCIAL 2\\PROYECTO\\etl_spark_parquet_advanced\\data\\raw\\bad_parquet\\ARROW-GH-41321.parquet',
   'download_status': 'SKIPPED_EXISTS',
   'sha256_file': '9539e0925ddb0b52a467cfe5535455be26869bb9bf3b5e48a255c7e2c1973537'},
  {'file_name': 'ARROW-GH-43605.parquet',
   'url': 'https://raw.githubusercontent.c

## 3. Inicialización de Spark

In [6]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("Python usado por Spark:", sys.executable)
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

Python usado por Spark: c:\Users\saray\AppData\Local\Programs\Python\Python314\python.exe
HADOOP_HOME: C:\hadoop


In [7]:
spark = get_spark_session(config)
spark.sparkContext.setLogLevel("WARN")
spark

## 4. Lectura segura individual y generación de inventario

Cada archivo se lee dentro de `try/except`. Si falla, el error queda en `error_message` y el pipeline continúa.

In [8]:
inventory_df = build_audit_file_inventory(spark, config, process_id)

# Validación de columnas exactas
expected_cols = [
    "process_id", "source_system", "service_type", "file_name", "file_path",
    "file_size_mb", "partition_year", "partition_month", "read_status",
    "record_count", "column_count", "schema_hash", "error_message", "processed_at"
]
assert inventory_df.columns == expected_cols, inventory_df.columns

inventory_df.show(50, truncate=80)

+------------------------------------+----------------------+------------+-----------------------------------+--------------------------------------------------------------------------------+------------+--------------+---------------+------------+------------+------------+----------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------+
|                          process_id|         source_system|service_type|                          file_name|                                                                       file_path|file_size_mb|partition_year|partition_month| read_status|record_count|column_count|                                                     schema_hash|                                                                   error_message|              processed_at|
+------------------------------------+----------------------+------------+------------------------------

In [9]:
audit_path = write_audit_inventory(inventory_df, config)
print("audit_file_inventory escrito en:", audit_path)

audit_file_inventory escrito en: C:\Users\saray\OneDrive\Documents\6to SEMESTRE\MODELADO\PARCIAL 2\PROYECTO\etl_spark_parquet_advanced\data\audit\audit_file_inventory


## 5. Resumen de estados de lectura

In [10]:
inventory_df.groupBy("source_system", "service_type", "read_status").count().orderBy("source_system", "service_type", "read_status").show(truncate=False)

+----------------------+------------+------------+-----+
|source_system         |service_type|read_status |count|
+----------------------+------------+------------+-----+
|APACHE_PARQUET_TESTING|bad_parquet |CORRUPT     |2    |
|APACHE_PARQUET_TESTING|bad_parquet |EMPTY       |1    |
|APACHE_PARQUET_TESTING|bad_parquet |SCHEMA_ERROR|2    |
|APACHE_PARQUET_TESTING|bad_parquet |SUCCESS     |4    |
|NYC_TLC               |fhvhv       |SUCCESS     |1    |
|NYC_TLC               |green       |SUCCESS     |2    |
|NYC_TLC               |yellow      |SUCCESS     |8    |
+----------------------+------------+------------+-----+



## 6. Validación de archivos corruptos, vacíos y exitosos

In [11]:
from pyspark.sql.functions import col

print("Archivos corruptos o con error:")
inventory_df.filter(col("read_status").isin("CORRUPT", "SCHEMA_ERROR")).select(
    "service_type", "file_name", "read_status", "error_message"
).show(30, truncate=100)

print("Archivos vacíos:")
inventory_df.filter(col("read_status") == "EMPTY").select(
    "service_type", "file_name", "record_count"
).show(30, truncate=False)

print("Archivos leídos correctamente:")
inventory_df.filter(col("read_status") == "SUCCESS").select(
    "service_type", "file_name", "record_count", "column_count", "schema_hash"
).show(30, truncate=80)

Archivos corruptos o con error:
+------------+----------------------+------------+----------------------------------------------------------------------------------------------------+
|service_type|             file_name| read_status|                                                                                       error_message|
+------------+----------------------+------------+----------------------------------------------------------------------------------------------------+
| bad_parquet|ARROW-GH-41317.parquet|     CORRUPT|             [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT32 (TIME(MILLIS,true)). SQLSTATE: 42846|
| bad_parquet|ARROW-GH-41321.parquet|     CORRUPT|             [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT32 (TIME(MILLIS,true)). SQLSTATE: 42846|
| bad_parquet|  PARQUET-1481.parquet|SCHEMA_ERROR|An error occurred while calling o139.count.\n: org.apache.spark.SparkException: [FAILED_READ_FILE...|
| bad_parquet|             README.md|SCHEMA_ERROR|An err

## 7. Lectura de carpetas particionadas completas

Esta sección demuestra que Spark puede leer carpetas por tipo de servicio usando particiones `year/month`.

In [12]:
from extract import read_partitioned_service_folder

for service in ["yellow", "green", "fhvhv"]:
    try:
        df_service = read_partitioned_service_folder(spark, config, service)
        print(f"Servicio: {service}")
        print("Columnas:", len(df_service.columns))
        df_service.printSchema()
    except Exception as e:
        print(f"No se pudo leer la carpeta completa de {service}: {e}")

Servicio: yellow
Columnas: 21
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

Servicio: green
Columnas

## 8. Evidencia de idempotencia

La tabla `audit_file_inventory` se escribe en modo `overwrite`, por eso una segunda ejecución reemplaza el resultado anterior y no duplica registros.

In [13]:
audit_check = spark.read.parquet(audit_path)
print("Registros en audit_file_inventory:", audit_check.count())
audit_check.groupBy("process_id").count().show(truncate=False)

Registros en audit_file_inventory: 20
+------------------------------------+-----+
|process_id                          |count|
+------------------------------------+-----+
|5c2747e7-dee3-467d-8766-55dafca011c7|20   |
+------------------------------------+-----+

